# Cost Analysis for Model Optimization

In this notebook, we'll analyze the cost implications of the various model optimization techniques we've explored. We'll calculate the ROI and payback period for each technique.

## 1. Import Dependencies

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

## 2. Load Metrics from Previous Notebooks

In [ ]:
# Load baseline metrics
with open('baseline_metrics.json', 'r') as f:
    baseline_metrics = json.load(f)

# Load quantized metrics if available
try:
    with open('quantized_metrics.json', 'r') as f:
        quantized_metrics = json.load(f)
    print("Loaded quantized metrics")
except FileNotFoundError:
    quantized_metrics = {}
    print("No quantized metrics found")

# Load pruned metrics if available
try:
    with open('pruned_metrics.json', 'r') as f:
        pruned_metrics = json.load(f)
    print("Loaded pruned metrics")
except FileNotFoundError:
    pruned_metrics = {}
    print("No pruned metrics found")

# Load deployment info if available
try:
    with open('deployment_info.json', 'r') as f:
        deployment_info = json.load(f)
    print("Loaded deployment info")
except FileNotFoundError:
    deployment_info = {}
    print("No deployment info found")

## 3. Define Cost Estimation Functions

In [ ]:
def estimate_monthly_cost(model_metrics, requests_per_month=1000000):
    """Estimate monthly cost for running a model in production."""
    # Assumptions
    compute_cost_per_hour = 0.5  # $0.5 per hour for compute (e.g., ml.g4dn.xlarge)
    storage_cost_per_gb_month = 0.023  # $0.023 per GB-month for S3
    
    # Calculate compute cost
    inference_time_hours = (model_metrics["inference_time"] * requests_per_month) / (1000 * 60 * 60)
    compute_cost = inference_time_hours * compute_cost_per_hour
    
    # Calculate storage cost
    storage_cost = (model_metrics["model_size"] / 1024) * storage_cost_per_gb_month
    
    # Total cost
    total_cost = compute_cost + storage_cost
    
    return {
        "compute_cost": compute_cost,
        "storage_cost": storage_cost,
        "total_cost": total_cost
    }

def calculate_roi(baseline_cost, optimized_cost, implementation_cost, months=12):
    """Calculate ROI for an optimization technique."""
    monthly_savings = baseline_cost - optimized_cost
    total_savings = monthly_savings * months
    roi = (total_savings - implementation_cost) / implementation_cost if implementation_cost > 0 else float('inf')
    payback_period = implementation_cost / monthly_savings if monthly_savings > 0 else float('inf')
    
    return {
        "monthly_savings": monthly_savings,
        "total_savings": total_savings,
        "roi": roi,
        "payback_period": payback_period
    }

## 4. Define Implementation Costs

Let's define estimated implementation costs for each optimization technique. These costs include engineering time and compute resources needed to implement the technique.

In [ ]:
# Define implementation costs for each technique
implementation_costs = {
    "quantization": {
        "engineering_hours": 8,  # Hours of engineering time
        "compute_hours": 2,      # Hours of compute time
        "hourly_rate": 100,      # Engineering hourly rate
        "compute_rate": 0.5      # Compute hourly rate
    },
    "pruning": {
        "engineering_hours": 16,  # Hours of engineering time
        "compute_hours": 8,       # Hours of compute time
        "hourly_rate": 100,       # Engineering hourly rate
        "compute_rate": 0.5       # Compute hourly rate
    }
}

# Calculate total implementation costs
for technique, costs in implementation_costs.items():
    engineering_cost = costs["engineering_hours"] * costs["hourly_rate"]
    compute_cost = costs["compute_hours"] * costs["compute_rate"]
    total_cost = engineering_cost + compute_cost
    implementation_costs[technique]["total_cost"] = total_cost
    
    print(f"{technique.capitalize()} implementation cost: ${total_cost:.2f}")

## 5. Calculate Monthly Costs for Each Model and Technique

In [ ]:
# Calculate monthly costs for each model and technique
monthly_costs = {}

# Define request volumes to analyze
request_volumes = [100000, 1000000, 10000000]  # 100K, 1M, 10M requests per month

for model_key in baseline_metrics.keys():
    monthly_costs[model_key] = {}
    
    # Calculate costs for different request volumes
    for requests in request_volumes:
        monthly_costs[model_key][requests] = {
            "baseline": estimate_monthly_cost(baseline_metrics[model_key], requests)["total_cost"]
        }
        
        # Add quantized costs if available
        if model_key in quantized_metrics:
            monthly_costs[model_key][requests]["quantized"] = estimate_monthly_cost(quantized_metrics[model_key], requests)["total_cost"]
        
        # Add pruned costs if available
        if model_key in pruned_metrics:
            monthly_costs[model_key][requests]["pruned"] = estimate_monthly_cost(pruned_metrics[model_key], requests)["total_cost"]

## 6. Calculate ROI and Payback Period

In [ ]:
# Calculate ROI and payback period for each model and technique
roi_data = []

for model_key in monthly_costs.keys():
    model_name = baseline_metrics[model_key]["model_name"]
    
    for requests in request_volumes:
        baseline_cost = monthly_costs[model_key][requests]["baseline"]
        
        # Calculate ROI for quantization if available
        if "quantized" in monthly_costs[model_key][requests]:
            quantized_cost = monthly_costs[model_key][requests]["quantized"]
            quantized_roi = calculate_roi(baseline_cost, quantized_cost, implementation_costs["quantization"]["total_cost"])
            
            roi_data.append({
                "Model": model_name,
                "Technique": "Quantization",
                "Requests": f"{requests/1000:.0f}K",
                "Monthly Savings ($)": quantized_roi["monthly_savings"],
                "Annual Savings ($)": quantized_roi["total_savings"],
                "ROI (1 year)": quantized_roi["roi"],
                "Payback Period (months)": quantized_roi["payback_period"]
            })
        
        # Calculate ROI for pruning if available
        if "pruned" in monthly_costs[model_key][requests]:
            pruned_cost = monthly_costs[model_key][requests]["pruned"]
            pruned_roi = calculate_roi(baseline_cost, pruned_cost, implementation_costs["pruning"]["total_cost"])
            
            roi_data.append({
                "Model": model_name,
                "Technique": "Pruning",
                "Requests": f"{requests/1000:.0f}K",
                "Monthly Savings ($)": pruned_roi["monthly_savings"],
                "Annual Savings ($)": pruned_roi["total_savings"],
                "ROI (1 year)": pruned_roi["roi"],
                "Payback Period (months)": pruned_roi["payback_period"]
            })

# Create DataFrame
roi_df = pd.DataFrame(roi_data)
roi_df

## 7. Visualize ROI and Payback Period

In [ ]:
# Filter data for 1M requests
roi_1m = roi_df[roi_df["Requests"] == "1000K"]

# Plot ROI
plt.figure(figsize=(12, 6))
sns.barplot(x="Model", y="ROI (1 year)", hue="Technique", data=roi_1m)
plt.title("1-Year ROI by Model and Technique (1M requests/month)")
plt.ylabel("ROI (Return on Investment)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Plot payback period
plt.figure(figsize=(12, 6))
sns.barplot(x="Model", y="Payback Period (months)", hue="Technique", data=roi_1m)
plt.title("Payback Period by Model and Technique (1M requests/month)")
plt.ylabel("Payback Period (months)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. Analyze Cost Savings by Request Volume

In [ ]:
# Create a pivot table to analyze savings by request volume
pivot_df = pd.pivot_table(
    roi_df,
    values="Monthly Savings ($)",
    index=["Model", "Technique"],
    columns="Requests",
    aggfunc=np.mean
)

pivot_df

In [ ]:
# Plot monthly savings by request volume for a specific model
model_key = list(baseline_metrics.keys())[0]  # Choose the first model
model_name = baseline_metrics[model_key]["model_name"]

# Filter data for the selected model
model_roi = roi_df[roi_df["Model"] == model_name]

# Create a pivot table
model_pivot = pd.pivot_table(
    model_roi,
    values="Monthly Savings ($)",
    index="Technique",
    columns="Requests",
    aggfunc=np.mean
)

# Plot
model_pivot.plot(kind="bar", figsize=(10, 6))
plt.title(f"Monthly Savings by Request Volume for {model_name}")
plt.ylabel("Monthly Savings ($)")
plt.xlabel("Optimization Technique")
plt.tight_layout()
plt.show()

## 9. Calculate Break-Even Point

In [ ]:
def calculate_break_even_requests(baseline_metrics, optimized_metrics, implementation_cost):
    """Calculate the number of requests needed to break even on implementation cost."""
    # Cost per request
    baseline_inference_time = baseline_metrics["inference_time"] / 1000  # Convert to seconds
    optimized_inference_time = optimized_metrics["inference_time"] / 1000  # Convert to seconds
    
    # Cost per hour of compute
    compute_cost_per_hour = 0.5  # $0.5 per hour
    
    # Cost per request
    baseline_cost_per_request = baseline_inference_time * (compute_cost_per_hour / 3600)
    optimized_cost_per_request = optimized_inference_time * (compute_cost_per_hour / 3600)
    
    # Savings per request
    savings_per_request = baseline_cost_per_request - optimized_cost_per_request
    
    # Break-even requests
    if savings_per_request > 0:
        break_even_requests = implementation_cost / savings_per_request
    else:
        break_even_requests = float('inf')
    
    return break_even_requests

In [ ]:
# Calculate break-even points
break_even_data = []

for model_key in baseline_metrics.keys():
    model_name = baseline_metrics[model_key]["model_name"]
    
    # Calculate break-even for quantization if available
    if model_key in quantized_metrics:
        break_even_requests = calculate_break_even_requests(
            baseline_metrics[model_key],
            quantized_metrics[model_key],
            implementation_costs["quantization"]["total_cost"]
        )
        
        break_even_data.append({
            "Model": model_name,
            "Technique": "Quantization",
            "Break-Even Requests": break_even_requests,
            "Break-Even Requests (M)": break_even_requests / 1000000
        })
    
    # Calculate break-even for pruning if available
    if model_key in pruned_metrics:
        break_even_requests = calculate_break_even_requests(
            baseline_metrics[model_key],
            pruned_metrics[model_key],
            implementation_costs["pruning"]["total_cost"]
        )
        
        break_even_data.append({
            "Model": model_name,
            "Technique": "Pruning",
            "Break-Even Requests": break_even_requests,
            "Break-Even Requests (M)": break_even_requests / 1000000
        })

# Create DataFrame
break_even_df = pd.DataFrame(break_even_data)
break_even_df

In [ ]:
# Plot break-even points
plt.figure(figsize=(12, 6))
sns.barplot(x="Model", y="Break-Even Requests (M)", hue="Technique", data=break_even_df)
plt.title("Break-Even Point by Model and Technique")
plt.ylabel("Break-Even Requests (millions)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 10. Create Cost Optimization Recommendations

In [ ]:
def generate_recommendations(model_key, baseline_metrics, quantized_metrics, pruned_metrics, roi_df):
    """Generate cost optimization recommendations for a model."""
    model_name = baseline_metrics[model_key]["model_name"]
    task = baseline_metrics[model_key]["task"]
    
    # Filter ROI data for this model and 1M requests
    model_roi = roi_df[(roi_df["Model"] == model_name) & (roi_df["Requests"] == "1000K")]
    
    # Determine best technique based on ROI
    if not model_roi.empty:
        best_technique = model_roi.loc[model_roi["ROI (1 year)"].idxmax()]
        best_technique_name = best_technique["Technique"]
        best_roi = best_technique["ROI (1 year)"]
        best_payback = best_technique["Payback Period (months)"]
        
        recommendations = {
            "model_name": model_name,
            "task": task,
            "best_technique": best_technique_name,
            "roi": best_roi,
            "payback_period": best_payback,
            "recommendations": []
        }
        
        # Add specific recommendations
        if best_payback < 1:
            recommendations["recommendations"].append(f"Implement {best_technique_name.lower()} immediately for quick ROI")
        elif best_payback < 3:
            recommendations["recommendations"].append(f"Implement {best_technique_name.lower()} for good medium-term ROI")
        else:
            recommendations["recommendations"].append(f"Consider {best_technique_name.lower()} for long-term cost savings")
        
        # Add technique-specific recommendations
        if best_technique_name == "Quantization":
            recommendations["recommendations"].append("Use dynamic quantization for minimal accuracy impact")
            recommendations["recommendations"].append("Consider quantization-aware training for better results")
        elif best_technique_name == "Pruning":
            recommendations["recommendations"].append("Experiment with different pruning ratios to find optimal balance")
            recommendations["recommendations"].append("Consider iterative pruning and fine-tuning for better accuracy")
        
        # Add general recommendations
        recommendations["recommendations"].append("Monitor inference performance after optimization")
        recommendations["recommendations"].append("Consider combining techniques for even better results")
        
        return recommendations
    else:
        return {
            "model_name": model_name,
            "task": task,
            "best_technique": "N/A",
            "roi": 0,
            "payback_period": float('inf'),
            "recommendations": ["Insufficient data for recommendations"]
        }

In [ ]:
# Generate recommendations for each model
recommendations = {}

for model_key in baseline_metrics.keys():
    recommendations[model_key] = generate_recommendations(
        model_key,
        baseline_metrics,
        quantized_metrics,
        pruned_metrics,
        roi_df
    )
    
    # Print recommendations
    print(f"Recommendations for {recommendations[model_key]['model_name']}:")
    print(f"Best technique: {recommendations[model_key]['best_technique']}")
    print(f"1-year ROI: {recommendations[model_key]['roi']:.2f}")
    print(f"Payback period: {recommendations[model_key]['payback_period']:.2f} months")
    print("Recommendations:")
    for rec in recommendations[model_key]['recommendations']:
        print(f"- {rec}")
    print()

## 11. Save Cost Analysis Results

In [ ]:
# Save cost analysis results
cost_analysis = {
    "monthly_costs": monthly_costs,
    "implementation_costs": implementation_costs,
    "recommendations": recommendations
}

with open('cost_analysis.json', 'w') as f:
    json.dump(cost_analysis, f, indent=2)

print("Cost analysis saved to cost_analysis.json")

## 12. Next Steps

In this notebook, we've analyzed the cost implications of various model optimization techniques. We've calculated ROI, payback periods, and break-even points, and generated recommendations for each model.

In the next notebook, we'll clean up the resources we've created during this workshop.